In [177]:
%load_ext autoreload
%autoreload 1
%aimport classes.GaloisField
%aimport classes.GolayDecoder

import numpy as np
from itertools import combinations

from classes.GaloisField import *
from classes.GaloisPoly  import *
from classes.GolayEncoder import GolayEncoder
from classes.GolayDecoder import GolayDecoder

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Generate Galois Field

In [178]:
gf              = GaloisField(1,0b11)
encoder_model   = GolayEncoder()
k               = encoder_model._k
n               = encoder_model._n


Field Closed Succesfully!, 1 Non-Zero Elements


## 1. Codewords Test

### 1.1 Generate all codewords

In [179]:
encoder_output = None
for w in range(2**k):
    w   = gf.do_unpack(w, bit_width=k)
    cw  = encoder_model.encode(w)
    encoder_output = cw if encoder_output is None else np.vstack((encoder_output, cw))

n_codewords = len(encoder_output)
n_codewords

4096

### 1.2 Decoding of codewords

In [180]:
n_codewords
#encoder_output 
decoder_model = GolayDecoder()

decoder_output = []

for i in range(n_codewords):
    decoder_output.append(decoder_model.correct(encoder_output[i]))

#print(decoder_output)

Field Closed Succesfully!, 1 Non-Zero Elements


### 1.3 Error calculation `o_err` (null, all zeros)

In [181]:
o_err           = []
corrected       = []
uncorrectable   = []

o_corrected     = []
o_uncorrectable = []
o_msg           = []

for i in range(n_codewords):
    decoded_codeword = decoder_output[i][0]
    o_err.append(decoded_codeword - encoder_output[i])

    o_msg.append(decoder_output[i][0][:12])
    o_corrected.append(int(decoder_output[i][1]))
    o_uncorrectable.append(int(decoder_output[i][2]))
    corrected.append(decoder_output[i][1])
    uncorrectable.append(decoder_output[i][2])

print(np.array(o_err).shape)
print(np.array(o_corrected).shape)
print(np.array(o_uncorrectable).shape)
print(np.array(o_msg).shape)

(4096, 24)
(4096,)
(4096,)
(4096, 12)


## 2. Codewords with errors test

### 2.1 Generate errors

In [182]:
def write_codewords_with_error(filename, codeword, max_errors):
    """
    Genera todos los vectores de 24 bits que se obtienen al introducir
    de 0 a max_errors errores en codeword (peso 0, 1, 2, ..., max_errors).

    Escribe una línea por vector, con las columnas separadas por espacio:
        1. receive_word  (24 bits)
        2. word          (12 bits, info bits decodificados)
        3. error_mask    (24 bits, errores introducidos)
        4. corrected     (0/1)
        5. uncorrectable (0/1)
    """

    n = len(codeword)

    # Convertir codeword [0,1,1,...] -> entero
    codeword_int = int("".join(map(str, codeword)), 2)

    # Todas las posiciones posibles donde colocar los errores, por peso
    # C(24,0) + C(24,1) + C(24,2) + C(24,3) + C(24,4) = 12951
    error_masks = [
        sum(1 << (n - 1 - pos) for pos in error_positions)
        for n_errors in range(max_errors + 1)
        for error_positions in combinations(range(n), n_errors)
    ]

    with open(filename, "w") as file:
        for mask in error_masks:
            received_word = codeword_int ^ mask
            received_bits = np.array(
                [int(bit) for bit in format(received_word, f"0{n}b")], dtype=np.uint8)

            word, corrected, uncorrectable = decoder_model.decode(received_bits)

            file.write(" ".join([
                format(received_word, f"0{n}b"),
                "".join(map(str, word)),
                format(mask, f"0{n}b"),
                str(int(corrected)),
                str(int(uncorrectable)),
            ]) + "\n")

    return len(error_masks)

max_errors = 4 # 4 errors limit
filename = "outputs/decoder/codeword_with_errors.txt"
n_lines = write_codewords_with_error(filename, encoder_output[3838], max_errors)
print(f"{n_lines} codewords written to {filename}")

12951 codewords written to outputs/decoder/codeword_with_errors.txt


## 3. Golay (24,12) decoding example

In [183]:
r = encoder_output[3576]
# Generate random error for r (word received/transmitted)
r = r ^ np.array([
    [1,0,0,1,0,0,0,1,0,0,0,0]   ,
    [0,0,0,0,0,0,0,0,0,0,0,0]   ]).flatten()

decoder_model = GolayDecoder()

w, corrected, uncorrectable = decoder_model.decode(r)

# r: word with errors
# word decoded (possible codeword),
# flags (corrected, uncorrectable)
# encoder output (word received transmitted)
r, decoder_model.decode(r), encoder_output[3576]

Field Closed Succesfully!, 1 Non-Zero Elements


(array([0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1]),
 (array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0]), True, False),
 array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
        1, 1], dtype=uint8))

In [184]:
# decode all codewords, no errors
for cw in encoder_output:
    w, corrected, uncorrectable = decoder_model.decode(cw, full_codeword=True)
    assert(np.all(w == cw))
    assert(not corrected and not uncorrectable)

for cw in encoder_output:
    error_seed      = np.random.randint(0, 0b11111)
    # decimal error_seed converted into n-bits
    error           = decoder_model._gf.do_unpack(error_seed, bit_width=decoder_model._n)
    # calculate hamming weight
    error_weight    = decoder_model._gf.hamming_weight(error)
    
    np.random.shuffle(error)
    w, corrected, uncorrectable = decoder_model.decode(cw ^ error, full_codeword=True)
    
    # no errors
    if error_weight == 0:
        assert(np.all(w == cw))
        assert(not corrected and not uncorrectable)
    # 1 to 3 errors
    elif 1 <= error_weight <= 3:
        assert(np.all(w == cw))
        assert(corrected and not uncorrectable)
    # 4 errors
    else:
        assert(not np.all(w == cw))
        assert(not corrected and uncorrectable)

    # five or more errors this decoding fails, recovered bits and flags are invalid


## 4. Decoding test with selected values 

Verify the following values:

$$ r_{1} = 0xA5D9A6 $$
$$ r_{2} = 0xA5F9A4 $$
$$ r_{3} = 0xA5C9AA $$

### 4.1 Obtain values corrected.

In [185]:
import numpy as np

rx_test = [0xA5D9A6, 0xA5F9A4, 0xA5C9AA]

rx_test_array = [
    np.array([int(bit) for bit in format(x, '024b')], dtype=np.uint8)
    for x in rx_test]

rx_test_array

decoded_rx_test        = []
decoded_rx_test_binary = []

for i in range(len(rx_test_array)):
    decoded_rx_test.append(decoder_model.decode(rx_test_array[i], False))

# print(decoded_rx_test)

for decoded, valid, uncorrectable in decoded_rx_test:
    decoded_rx_test_binary.append(
        (decoded, int(valid), int(uncorrectable))
    )

# decoded word, o_corrected, o_uncorrected
decoded_rx_test_binary

[(array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 1, 0),
 (array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 1, 0),
 (array([1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0], dtype=uint8), 0, 1)]

### 4.2 Obtain mask error

In [186]:
s_q_vectors  = []
error_vector = []

for i in range (len(rx_test_array)):
    s_q_vectors.append(decoder_model.get_s_q(rx_test_array[i]))
    error_vector.append(decoder_model.get_error(s_q_vectors[i][0], s_q_vectors[i][1]))
    if error_vector[i] is None:
        error_vector[i] = [0]*24

# One line per test vector, columns separated by space:
#   1. rx_test        (24 bits)
#   2. msg            (12 bits, decoded word)
#   3. error          (24 bits, error mask)
#   4. corrected      (0/1)
#   5. uncorrectable  (0/1)
filename = "outputs/decoder/error_mask_test.txt"
with open(filename, "w") as file:
    for rx, (msg, corrected, uncorrectable), err in zip(rx_test_array, decoded_rx_test_binary, error_vector):
        file.write(" ".join([
            "".join(map(str, rx)),
            "".join(map(str, msg)),
            "".join(map(str, err)),
            str(int(corrected)),
            str(int(uncorrectable)),
        ]) + "\n")

error_vector

[array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1], dtype=uint8),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1], dtype=uint8),
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]

## 5. Write `one_codeword_full_errors_and_three_test_vectors.txt` file

In [187]:
# One codeword with errors of weight 0..4 (section 2.1, 12951 lines)
# followed by the three selected test vectors (section 4.2, 3 lines)
# Columns: rx msg error corrected uncorrectable
input_filenames = [
    "outputs/decoder/codeword_with_errors.txt",
    "outputs/decoder/error_mask_test.txt",
]
output_filename = "outputs/decoder/one_codeword_full_errors_and_three_test_vectors.txt"

n_lines = 0
with open(output_filename, "w") as out_file:
    for input_filename in input_filenames:
        with open(input_filename) as in_file:
            for line in in_file:
                out_file.write(line)
                n_lines += 1

print(f"{n_lines} lines written to {output_filename}")

12954 lines written to outputs/decoder/one_codeword_full_errors_and_three_test_vectors.txt
